In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    markers = (
        "requirements-runpod-cu124.txt",
        "defapi_qwen2_5_coder_14b_lora.yaml",
        "defapi_qwen2_5_coder_14b_lora.ipynb",
    )
    for candidate in (start, *start.parents):
        if any((candidate / marker).exists() for marker in markers):
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REQUIREMENTS_PATH = PROJECT_ROOT / "requirements-runpod-cu124.txt"
EXPECTED_VERSIONS = [
    ("torch", "2.5.1+cu124"),
    ("transformers", "4.46.3"),
    ("accelerate", "1.1.1"),
    ("datasets", "3.1.0"),
    ("peft", "0.13.2"),
    ("huggingface-hub", "0.26.5"),
    ("tokenizers", "0.20.3"),
    ("safetensors", "0.4.5"),
    ("sentencepiece", "0.2.0"),
    ("protobuf", "5.28.3"),
    ("PyYAML", "6.0.2"),
    ("wandb", "0.18.7"),
    ("packaging", "24.2"),
]


def installed_version(distribution: str) -> str | None:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


if REQUIREMENTS_PATH.exists():
    mismatches = []
    for distribution, expected_version in EXPECTED_VERSIONS:
        current_version = installed_version(distribution)
        if current_version != expected_version:
            mismatches.append((distribution, current_version, expected_version))

    if mismatches:
        print("Installing locked RunPod package stack:")
        for distribution, current_version, expected_version in mismatches:
            print(f"- {distribution}: {current_version or 'missing'} -> {expected_version}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "--no-cache-dir",
            "-r",
            str(REQUIREMENTS_PATH),
        ])
    else:
        print("Locked RunPod package stack is already installed.")
else:
    print(f"WARNING: {REQUIREMENTS_PATH} not found; skipping locked package install.")
    print("If imports fail later, upload requirements-runpod-cu124.txt or install requirements manually.")

print(f"Project root: {PROJECT_ROOT}")


## 2. Environment Check


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Platform: {platform.platform()}")
print(f"Working directory: {Path.cwd()}")

try:
    import torch
    print(f"torch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA runtime: {torch.version.cuda}")
        for index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(index)
            total_gb = props.total_memory / 1024**3
            print(f"GPU {index}: {props.name} ({total_gb:.1f} GB VRAM)")
    else:
        print("GPU: not available")
except (ImportError, RuntimeError) as exc:
    print(f"torch check failed: {exc}")

try:
    result = subprocess.run(["nvidia-smi"], check=False, text=True, capture_output=True)
    print(result.stdout[:2000] if result.stdout else "nvidia-smi produced no stdout")
except FileNotFoundError:
    print("nvidia-smi not found")

## 3. Imports


In [ ]:
import inspect
import os
import sys
from pathlib import Path
from pprint import pprint
from typing import Any, Mapping

os.environ.setdefault("HF_HOME", "/workspace/.cache/huggingface")
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.cache/huggingface/transformers")
os.environ.setdefault("HF_DATASETS_CACHE", "/workspace/.cache/huggingface/datasets")
HF_TOKEN = os.environ.get("HF_TOKEN") or None

import yaml
import torch
from datasets import DatasetDict, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, set_seed

try:
    import wandb
except ModuleNotFoundError:
    wandb = None

print("imports complete")
print(f"HF_HOME={os.environ['HF_HOME']}")


## 4. Config

Edit this dictionary directly in Jupyter. The default values are full training settings for A100 80GB.


In [ ]:
CONFIG: dict[str, Any] = {
    "model_name": "Qwen/Qwen2.5-Coder-14B-Instruct",
    "dataset_name": "hitoshura25/crossvul",
    "dataset_split": "train",
    "eval_split": "validation",
    "output_dir": "/workspace/checkpoints/defapi-qwen2.5-coder-14b-lora",
    "max_seq_length": 2048,
    "attn_implementation": "sdpa",
    "gradient_checkpointing": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "save_steps": 100,
    "eval_steps": 100,
    "logging_steps": 10,
    "save_total_limit": 2,
    "report_to": "none",
    "wandb_project": "defapi-finetuning",
    "wandb_run_name": "qwen2.5-coder-14b-lora-full",
    "seed": 42,
}

set_seed(int(CONFIG["seed"]))

Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
pprint(CONFIG)


## 5. Hugging Face / W&B Login Guide

For private or gated Hugging Face datasets/models, run `huggingface-cli login` in a Jupyter terminal or use `notebook_login()`. W&B login runs only when `CONFIG["report_to"] == "wandb"`.


In [ ]:
# Hugging Face login options:
# 1. Jupyter Terminal: huggingface-cli login
# 2. Notebook:
# from huggingface_hub import notebook_login
# notebook_login()

if CONFIG["report_to"] == "wandb":
    if wandb is None:
        raise ModuleNotFoundError("wandb is not installed. Run the locked package install cell or set report_to='none'.")
    os.environ.setdefault("WANDB_PROJECT", CONFIG["wandb_project"])
    wandb.login()
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled because CONFIG['report_to'] is 'none'.")


## 6. Dataset Load

Supports Hugging Face split slicing such as `train[:100]`. If no eval split is configured, the notebook creates an eval split with `train_test_split(test_size=0.1)`. The default dataset is `hitoshura25/crossvul`.


In [ ]:
def load_defapi_dataset(config: Mapping[str, Any]) -> DatasetDict:
    dataset_name = str(config["dataset_name"])
    if dataset_name.startswith("<"):
        raise ValueError("Set CONFIG['dataset_name'] to a real Hugging Face dataset name before running this cell.")

    train_dataset = load_dataset(dataset_name, split=config["dataset_split"], token=HF_TOKEN)
    eval_split = config.get("eval_split")

    if eval_split:
        eval_dataset = load_dataset(dataset_name, split=eval_split, token=HF_TOKEN)
        return DatasetDict({"train": train_dataset, "eval": eval_dataset})

    if len(train_dataset) < 2:
        raise ValueError("At least 2 rows are required to auto-create an eval split.")

    split = train_dataset.train_test_split(test_size=0.1, seed=int(config["seed"]), shuffle=True)
    return DatasetDict({"train": split["train"], "eval": split["test"]})


dataset = load_defapi_dataset(CONFIG)
print(dataset)
print("train columns:", dataset["train"].column_names)
print("eval columns:", dataset["eval"].column_names)


## 7. Dataset Validation

Rows are converted to chat messages for SFT. CrossVul metadata such as `CWE:`, `Description:`, `Language:`, and `Vulnerable code:` is not used as assistant target text; the user message asks for a repair, and the assistant message starts with fixed code plus a short explanation.


In [ ]:
from dataclasses import dataclass

SYSTEM_MESSAGE = "You are a secure coding assistant. Fix vulnerable code with safe, practical changes."

REQUIRED_FIELDS = ("instruction", "input", "output")
CROSSVUL_FIELDS = ("cwe_id", "cwe_description", "language", "vulnerable_code", "fixed_code")
LANGUAGE_TAG_ALIASES = {
    "c++": "cpp",
    "c#": "csharp",
    "javascript": "javascript",
    "typescript": "typescript",
    "python": "python",
    "java": "java",
    "go": "go",
    "ruby": "ruby",
    "php": "php",
}
FORBIDDEN_TARGET_MARKERS = (
    "작업:",
    "출력:",
    "CWE:",
    "Description:",
    "Language:",
    "Vulnerable code:",
)


def has_fields(example: Mapping[str, Any], fields: tuple[str, ...]) -> bool:
    return all(field in example for field in fields)


def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    return str(value).strip()


def language_tag_for(language: str) -> str:
    normalized = language.strip().lower()
    return LANGUAGE_TAG_ALIASES.get(normalized, normalized.replace(" ", "-") or "text")


def repair_user_prompt(language: str, vulnerable_code: str) -> str:
    language_tag = language_tag_for(language)
    return (
        f"Fix this vulnerable {language or 'code'} code and explain briefly:\n\n"
        f"```{language_tag}\n{vulnerable_code}\n```"
    )


def repair_assistant_response(language: str, fixed_code: str, description: str) -> str:
    language_tag = language_tag_for(language)
    explanation = description or (
        "The original code handles untrusted input unsafely. The fix uses safer APIs and "
        "keeps user-controlled data out of dangerous execution or access paths."
    )
    return (
        f"```{language_tag}\n{fixed_code}\n```\n\n"
        f"{explanation}\n\n"
        "The important security property is that untrusted input is handled as data, "
        "not as executable code, query syntax, filesystem traversal, or a secret literal."
    )


def build_messages(example: Mapping[str, Any]) -> list[dict[str, str]]:
    """Build chat SFT messages with only the requested repair task in the prompt."""
    if has_fields(example, REQUIRED_FIELDS):
        instruction = normalize_text(example["instruction"])
        input_text = normalize_text(example["input"])
        output = normalize_text(example["output"])
        user_content = f"{instruction}\n\n{input_text}".strip()
        assistant_content = output
    elif has_fields(example, CROSSVUL_FIELDS):
        language = normalize_text(example["language"])
        vulnerable_code = normalize_text(example["vulnerable_code"])
        fixed_code = normalize_text(example["fixed_code"])
        description = normalize_text(example["cwe_description"])
        user_content = repair_user_prompt(language, vulnerable_code)
        assistant_content = repair_assistant_response(language, fixed_code, description)
    else:
        available = ", ".join(example.keys())
        raise ValueError(
            "Dataset row must contain either instruction/input/output fields "
            f"or CrossVul fields. Available columns: {available}."
        )

    if not user_content or not assistant_content:
        raise ValueError("Dataset row produced an empty user prompt or assistant response.")

    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content},
    ]


def _chat_template_ids(tokenizer: Any, messages: list[dict[str, str]], add_generation_prompt: bool) -> list[int]:
    if hasattr(tokenizer, "apply_chat_template"):
        ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=add_generation_prompt,
            return_tensors=None,
        )
        if ids and isinstance(ids[0], list):
            return list(ids[0])
        return list(ids)

    text = "\n".join(f"<{message['role']}>\n{message['content']}" for message in messages)
    if add_generation_prompt:
        text = f"{text}\n<assistant>\n"
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def tokenize_chat_example(
    example: Mapping[str, Any],
    tokenizer: Any,
    max_seq_length: int,
) -> dict[str, list[int]]:
    messages = build_messages(example)
    prompt_ids = _chat_template_ids(tokenizer, messages[:-1], add_generation_prompt=True)
    full_ids = _chat_template_ids(tokenizer, messages, add_generation_prompt=False)

    if len(full_ids) <= len(prompt_ids):
        raise ValueError("Assistant response was fully truncated or chat template prefix did not match.")

    input_ids = full_ids[:max_seq_length]
    labels = [-100] * min(len(prompt_ids), len(input_ids))
    labels.extend(input_ids[len(labels):])
    labels = labels[: len(input_ids)]

    if all(label == -100 for label in labels):
        raise ValueError("All labels are masked. Increase max_seq_length or shorten prompts.")

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
    }


def tokenize_chat_batch(
    batch: Mapping[str, list[Any]],
    tokenizer: Any,
    max_seq_length: int,
) -> dict[str, list[list[int]]]:
    keys = list(batch.keys())
    rows = [dict(zip(keys, values, strict=True)) for values in zip(*(batch[key] for key in keys), strict=True)]
    tokenized = [tokenize_chat_example(row, tokenizer, max_seq_length) for row in rows]
    return {
        "input_ids": [row["input_ids"] for row in tokenized],
        "attention_mask": [row["attention_mask"] for row in tokenized],
        "labels": [row["labels"] for row in tokenized],
    }


@dataclass
class DataCollatorForAssistantOnlyLM:
    tokenizer: Any
    label_pad_token_id: int = -100

    def __call__(self, features: list[dict[str, list[int]]]) -> dict[str, Any]:
        labels = [feature["labels"] for feature in features]
        input_features = [
            {key: value for key, value in feature.items() if key != "labels"}
            for feature in features
        ]
        batch = self.tokenizer.pad(input_features, padding=True, return_tensors="pt")
        max_length = batch["input_ids"].shape[1]
        padded_labels = []
        for label in labels:
            pad_length = max_length - len(label)
            if self.tokenizer.padding_side == "left":
                padded = [self.label_pad_token_id] * pad_length + label
            else:
                padded = label + [self.label_pad_token_id] * pad_length
            padded_labels.append(padded)
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch


def debug_print_label_mask(dataset: Any, tokenizer: Any, n: int = 5) -> None:
    count = min(n, len(dataset))
    for index in range(count):
        example = dataset[index]
        visible_labels = [token_id for token_id in example["labels"] if token_id != -100]
        masked = len(example["labels"]) - len(visible_labels)
        target_text = tokenizer.decode(visible_labels, skip_special_tokens=True)
        print("=" * 80)
        print(f"sample={index} input_tokens={len(example['input_ids'])} masked_labels={masked} target_tokens={len(visible_labels)}")
        print(target_text[:2000])
        leaked_markers = [marker for marker in FORBIDDEN_TARGET_MARKERS if marker in target_text]
        if leaked_markers:
            raise ValueError(f"Forbidden prompt markers leaked into assistant labels: {leaked_markers}")


def validate_dataset(dataset_dict: DatasetDict) -> None:
    for split_name, split_dataset in dataset_dict.items():
        if len(split_dataset) == 0:
            raise ValueError(f"{split_name} split is empty.")
        messages = build_messages(split_dataset[0])
        roles = [message["role"] for message in messages]
        if roles != ["system", "user", "assistant"]:
            raise ValueError(f"Unexpected chat roles for {split_name}: {roles}")
        print(f"{split_name}: {len(split_dataset)} rows validated as chat SFT messages.")


validate_dataset(dataset)


## 8. Chat Sample Preview

Inspect the actual chat messages before tokenization. The assistant content should contain only the desired answer, not dataset field labels.


In [ ]:
sample_messages = build_messages(dataset["train"][0])
for message in sample_messages:
    print("=" * 80)
    print(message["role"])
    print(message["content"][:2000])


## 9. Tokenizer and Assistant-only Label Masking

The tokenized dataset includes `labels`. User/system prompt tokens are masked with `-100`; only assistant response tokens keep their labels and contribute to loss.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print(f"pad_token={tokenizer.pad_token!r}, eos_token={tokenizer.eos_token!r}, padding_side={tokenizer.padding_side}")


def tokenize_chat_batch_for_map(batch: Mapping[str, list[Any]]) -> dict[str, list[list[int]]]:
    return tokenize_chat_batch(batch, tokenizer=tokenizer, max_seq_length=int(CONFIG["max_seq_length"]))


tokenized_dataset = dataset.map(
    tokenize_chat_batch_for_map,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing DefAPI chat samples with assistant-only labels",
)

print("tokenized columns:", tokenized_dataset["train"].column_names)
print("first tokenized length:", len(tokenized_dataset["train"][0]["input_ids"]))
debug_print_label_mask(tokenized_dataset["train"], tokenizer, n=5)


## 10. Model Load


In [ ]:
BF16_SUPPORTED = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
TRAIN_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
print(f"bf16 supported: {BF16_SUPPORTED}; model torch_dtype: {TRAIN_DTYPE}")

model_load_kwargs: dict[str, Any] = {
    "device_map": "auto",
    "torch_dtype": TRAIN_DTYPE,
    "trust_remote_code": True,
    "attn_implementation": CONFIG["attn_implementation"],
    "token": HF_TOKEN,
}

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    **model_load_kwargs,
)
model.config.use_cache = False
print(f"model loaded with attention: {model.config._attn_implementation}")


## 11. LoRA Config


In [ ]:
lora_config = LoraConfig(
    r=int(CONFIG["lora_r"]),
    lora_alpha=int(CONFIG["lora_alpha"]),
    lora_dropout=float(CONFIG["lora_dropout"]),
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
lora_config


## 12. TrainingArguments

Handles `eval_strategy` vs `evaluation_strategy` across Transformers versions. The notebook tokenizes the `text` column explicitly before training, so the trainer receives tensor-ready `input_ids` and `attention_mask` columns.


In [ ]:
def supports_parameter(cls: type, parameter_name: str) -> bool:
    return parameter_name in inspect.signature(cls.__init__).parameters


def supported_init_kwargs(cls: type, kwargs: dict[str, Any]) -> dict[str, Any]:
    signature = inspect.signature(cls.__init__)
    if any(parameter.kind == inspect.Parameter.VAR_KEYWORD for parameter in signature.parameters.values()):
        return kwargs
    return {key: value for key, value in kwargs.items() if key in signature.parameters}

training_kwargs: dict[str, Any] = {
    "output_dir": CONFIG["output_dir"],
    "run_name": CONFIG["wandb_run_name"],
    "report_to": CONFIG["report_to"],
    "num_train_epochs": CONFIG["num_train_epochs"],
    "per_device_train_batch_size": CONFIG["per_device_train_batch_size"],
    "per_device_eval_batch_size": CONFIG["per_device_train_batch_size"],
    "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
    "learning_rate": CONFIG["learning_rate"],
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "logging_steps": CONFIG["logging_steps"],
    "eval_steps": CONFIG["eval_steps"],
    "save_steps": CONFIG["save_steps"],
    "save_total_limit": CONFIG["save_total_limit"],
    "bf16": BF16_SUPPORTED,
    "fp16": bool(torch.cuda.is_available() and not BF16_SUPPORTED),
    "gradient_checkpointing": CONFIG["gradient_checkpointing"],
    "optim": "adamw_torch",
    "save_strategy": "steps",
    "logging_strategy": "steps",
    "remove_unused_columns": False,
}


strategy_name = "eval_strategy" if supports_parameter(TrainingArguments, "eval_strategy") else "evaluation_strategy"
training_kwargs[strategy_name] = "steps"

training_args = TrainingArguments(**supported_init_kwargs(TrainingArguments, training_kwargs))
training_args


## 13. Trainer

Uses a `transformers.Trainer` subclass that does not call `.to(device)`, because the model is already dispatched by `device_map="auto"`. The data collator pads existing assistant-only labels; it does not recreate labels from the full input.


In [ ]:
class NoMoveTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        return model


data_collator = DataCollatorForAssistantOnlyLM(tokenizer=tokenizer)

trainer = NoMoveTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    data_collator=data_collator,
)
trainer


## 14. Full Training Cell

Runs the full training config, saves the adapter/model state and tokenizer, and prints the checkpoint path.


In [ ]:
torch.cuda.empty_cache()
train_result = trainer.train()

output_dir = Path(CONFIG["output_dir"])
trainer.save_model(str(output_dir))
tokenizer.save_pretrained(str(output_dir))

print(train_result)
print(f"Adapter/model and tokenizer saved to: {output_dir}")
